In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
import ipywidgets as widgets
from IPython.display import display

In [ ]:
df_facts = pd.read_csv('steamUserFact.csv')
df_games = pd.read_csv('gameDim.csv', encoding='latin1')
df_merged = pd.merge(df_facts, df_games[['game_id', 'game_name']], on='game_id')
df_merged = df_merged.drop_duplicates(subset=['steam_id', 'game_name'])

In [ ]:
pivot_table = df_merged.pivot(index='game_name', columns='steam_id', values='playtime_hours').fillna(0)
game_matrix = csr_matrix(pivot_table.values)

In [ ]:
knn_model = NearestNeighbors(metric='cosine', algorithm='brute')
knn_model.fit(game_matrix)

In [ ]:
def get_recommendations(game_name):
        game_idx = pivot_table.index.get_loc(game_name)
        distances, indices = knn_model.kneighbors(pivot_table.iloc[game_idx, :].values.reshape(1, -1), n_neighbors=6)
        print(f"\nOthers who played '{game_name}' also played:")
        print("-" * 40)
        for i in range(1, len(distances.flatten())):
            recommended_game = pivot_table.index[indices.flatten()[i]]
            similarity = round((1 - distances.flatten()[i]) * 100, 2)
            print(f"{i}. {recommended_game} (Crowd Similarity: {similarity}%)")

In [ ]:
game_list = pivot_table.index.tolist()

text_input = widgets.Combobox(
    description='Search:',
    placeholder='Type a game name...',
    options=game_list,
    ensure_option=False
)

In [ ]:
button = widgets.Button(description='Find What Others Played', button_style='success', layout={'width': '200px'})
output = widgets.Output()

def on_button_clicked(b):
    with output:
        output.clear_output()
        user_search = text_input.value.strip()
        if not user_search:
            print("Please enter a game name to search for recommendations.")
            return

        case_match = pivot_table.index[pivot_table.index.str.lower() == user_search.lower()]
        if len(case_match) > 0:
            get_recommendations(case_match[0])
        else:
            print("\nGame not found in the user playtime database.\nPlease check your spelling or try a more popular game.")

button.on_click(on_button_clicked)

In [ ]:
display(widgets.HBox([text_input, button]), output)